<a href="https://colab.research.google.com/github/K1taK1ta/deep-learning-notebooks/blob/main/RNN2_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
import keras
import numpy as np
import pandas as pd
import transformers
import tensorflow as tf
import tqdm.notebook as tqdm
import matplotlib.pyplot as plt
import math

# Dataset

In [ ]:
from google.colab import drive
import os
import datetime

drive.mount('/content/drive/')

save_folder = '/content/drive/MyDrive/Colab Notebooks/NLP/RNN2'
folder_path_weight = save_folder + '/models/models_weight/'

os.makedirs(folder_path_weight, exist_ok=True)
filepath_best_without = os.path.join(folder_path_weight, "best_model_rnn2_without_att.keras")
filepath_best_with = os.path.join(folder_path_weight, "best_model_rnn2_with_att.keras")

HISTORY_DIR = save_folder + '/models/history/'
os.makedirs(HISTORY_DIR, exist_ok=True)
logdir = os.path.join(HISTORY_DIR, datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

dataset_path = save_folder + "/RNN_2_data.tsv"


Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
df = pd.read_csv(

    dataset_path,
    sep="\t",
    header=None,
    quoting=3, # если текст может содержать кавычки, и ты не хочешь, чтобы их трактовали как ограничители. Именно так часто делают для Tatoeba.

    )

print(df.head())

      0                                                  1       2  \
0   243  Один раз в жизни я делаю хорошее дело... И оно...    3257   
1  5409                      Давайте что-нибудь попробуем!    1276   
2  5410                               Мне пора идти спать.    1277   
3  5411                                    Что ты делаешь?   16492   
4  5411                                    Что ты делаешь?  511884   

                                                   3  
0  For once in my life I'm doing a good deed... A...  
1                               Let's try something.  
2                             I have to go to sleep.  
3                                What are you doing?  
4                                  What do you make?  


In [ ]:
df.columns = ["rus_id", "rus_text", "eng_id", "eng_text"]
df.shape

(782365, 4)

In [ ]:
class Tokenizer(transformers.GPT2Tokenizer):

    def build_inputs_with_special_tokens(self, token_ids_0, token_ids_1=None):
        if token_ids_1 is None:
            return [self.bos_token_id, *token_ids_0, self.eos_token_id]


        return [self.bos_token_id, *token_ids_0, self.bos_token_id, *token_ids_1, self.eos_token_id]

In [ ]:
tokenizer = Tokenizer.from_pretrained('gpt2')

tokenizer.add_special_tokens({'bos_token': '<BOS>', 'pad_token': '<PAD>'})
tokenizer.pad_token = '<PAD>'

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'Tokenizer'.


Делаем сплит

In [ ]:
from sklearn.model_selection import train_test_split
input_texts = df['rus_text'].tolist()
target_texts = df['eng_text'].tolist()

sizes_inp = [len(text) for text in input_texts]
sizes_tar = [len(text) for text in target_texts]


In [ ]:
# Выбираем длину, покрывающую 99% текстов
max_len_inp = int(np.percentile(sizes_inp, 99))
print("Выбранная max_len_inp:", max_len_inp)

max_len_tar = int(np.percentile(sizes_tar, 99))
print("Выбранная max_len_tar:", max_len_tar)


Выбранная max_len_inp: 80
Выбранная max_len_tar: 79


Вар 1

In [ ]:
input_ids = tokenizer(
            input_texts,
            max_length=max_len_inp,
            truncation=True,
            padding='max_length',
            return_tensors='np',
            add_special_tokens=False,
        )['input_ids']

target_ids = tokenizer(
            target_texts,
            max_length=max_len_tar,
            truncation=True,
            padding='max_length',
            return_tensors='np',
            add_special_tokens=True,
        )['input_ids']

Вар 2

In [ ]:
# input_ids = tokenizer(
#             input_texts,
#             max_length=max_len_inp,
#             truncation=True,
#             # padding='max_length',
#             return_tensors='np',
#             add_special_tokens=False,
#         )['input_ids']

# target_ids = tokenizer(
#             target_texts,
#             max_length=max_len_tar,
#             truncation=True,
#             # padding='max_length',
#             return_tensors='np',
#             add_special_tokens=True,
#         )['input_ids']

In [ ]:
input_train, input_test, target_train, target_test = train_test_split(
    input_ids, target_ids, test_size=0.2, random_state=42
)

In [ ]:
steps_per_epoch = math.ceil(len(input_train) / 64)
validation_steps = math.ceil(len(input_test) / 64)

## Создание батчей

Вар 1

In [ ]:
def create_tf_dataset_fixed(input_texts, target_texts, tokenizer, batch_size=64, repeat=True, shuffle=False):

    input_texts = tf.convert_to_tensor(input_texts, dtype=tf.int32)
    target_texts = tf.convert_to_tensor(target_texts, dtype=tf.int32)

    decoder_inputs = target_texts[:, :-1]
    decoder_targets = target_texts[:, 1:]

    mask = tf.cast(tf.not_equal(decoder_targets, tokenizer.pad_token_id), tf.float32)

    dataset = tf.data.Dataset.from_tensor_slices(
        ((input_texts, decoder_inputs), decoder_targets, mask)
    )

    if shuffle:
        dataset = dataset.shuffle(100000)

    if repeat:
        dataset = dataset.repeat()

    dataset = dataset.batch(batch_size)

    #    # явное приведение mask к float32 и targets к правильной форме
    # dataset = dataset.map(lambda x, y, w: (
    #     (x[0], x[1]),           # inputs
    #     tf.cast(y, tf.int32),   # targets
    #     tf.cast(w, tf.float32)  # sample_weight
    # ))


    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


In [ ]:
train_dataset_tf = create_tf_dataset_fixed(input_train, target_train, tokenizer, batch_size=64, repeat=True, shuffle=True)
test_dataset_tf = create_tf_dataset_fixed(input_test, target_test, tokenizer, batch_size=64, repeat=False)

Вар 2

In [ ]:
def create_tf_dataset_dynamic(input_texts, target_texts, tokenizer, batch_size=64, repeat=True, shuffle=False):

    # RaggedTensor позволяет хранить sequences разной длины
    input_texts = tf.ragged.constant(input_texts, dtype=tf.int32)
    target_texts = tf.ragged.constant(target_texts, dtype=tf.int32)

    dataset = tf.data.Dataset.from_tensor_slices((input_texts, target_texts))


    def preprocess(x, y):
        decoder_input = y[:-1]
        decoder_target = y[1:]
        mask = tf.cast(tf.not_equal(decoder_target, tokenizer.pad_token_id), tf.float32)
        return (x, decoder_input), decoder_target, mask

    dataset = dataset.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        dataset = dataset.shuffle(100_000,
                                  seed=42,
                                  reshuffle_each_iteration=repeat)

    # Паддинг только по батчу
    dataset = dataset.padded_batch(
        batch_size,
        padded_shapes=(
            ([None], [None]),  # input, decoder_input
            [None],            # decoder_target
            [None]             # mask
        ),
        padding_values=(
            (tokenizer.pad_token_id, tokenizer.pad_token_id),
            tokenizer.pad_token_id,
            0.0
        ),
        drop_remainder=True


        )

    dataset = dataset.cache()


    if repeat:
        dataset = dataset.repeat()

    return dataset


In [ ]:
train_dataset_tf = create_tf_dataset_dynamic(input_train, target_train, tokenizer, batch_size=64, repeat=False, shuffle=False)
test_dataset_tf = create_tf_dataset_dynamic(input_test, target_test, tokenizer, batch_size=64, repeat=False)

In [ ]:
train_dataset_tf = train_dataset_tf.shuffle(10000, seed=42).repeat().prefetch(tf.data.AUTOTUNE)
test_dataset_tf = test_dataset_tf.prefetch(tf.data.AUTOTUNE)

In [ ]:
# tf.keras.mixed_precision.set_global_policy("mixed_float16")
# tf.keras.mixed_precision.set_global_policy("float32")


# Model

In [ ]:
def get_name(prefix: str | None = None, suffix: str | None = None, separator: str = '_') -> str | None:
    return prefix and prefix + separator + suffix or suffix or None

In [ ]:
class MaskedEmbedding(tf.keras.layers.Layer):
    def __init__(self, input_dim, output_dim, mask_zero=False, mask_value=tokenizer.pad_token_id
, name=None, **kwargs):


        super().__init__(name=name, **kwargs)
        self.supports_masking = True
        self.mask_zero = mask_zero
        self.mask_value = mask_value

        self.embedding = tf.keras.layers.Embedding(
            input_dim=input_dim,
            output_dim=output_dim,
            mask_zero=mask_zero,
            name=f"{name}_embedding" if name is not None else None
        )


    def call(self, inputs, mask=None):
        return self.embedding(inputs)

    def compute_mask(self, inputs, mask=None):

        if self.mask_zero:
            return self.embedding.compute_mask(inputs)

        elif self.mask_value is not None:
            custom_mask = tf.keras.ops.not_equal(inputs, self.mask_value)
            return custom_mask

        return mask


In [ ]:
def prepare_encoder_states(encoder_states, cell_type, n_stacks, bidirectional):
    """
    Преобразует состояния энкодера для декодера.

    Arguments:
        encoder_states: tuple, состояния от encoder RNN
        cell_type: keras.layers.LSTMCell или GRUCell
        n_stacks: количество стэков в encoder
        bidirectional: True/False

    Returns:
        decoder_states: список тензоров для initial_state декодера
    """

    if issubclass(cell_type, keras.layers.LSTMCell):
        if bidirectional:

            forward_states = encoder_states[:n_stacks * 2]
            backward_states = encoder_states[n_stacks * 2:]

            b = backward_states
            f = forward_states

            lenght_h = len(backward_states)
            print(lenght_h)
            lenght_c = len(backward_states)
            print(lenght_c)

            print("Forward states:", forward_states)
            print("Backward states:", backward_states)

            concat_h = [keras.layers.Concatenate()([f[i], b[i]]) for i in range(0, lenght_h, 2)]
            concat_c = [keras.layers.Concatenate()([f[i], b[i]]) for i in range(1, lenght_c, 2)]

            print("Concatenated h:", concat_h)
            print("Concatenated c:", concat_c)

            decoder_states = []
            for h, c in zip(concat_h, concat_c):
                decoder_states +=[h, c]

            print("Final decoder_states:", decoder_states)
            return decoder_states
        else:
            decoder_states = encoder_states
            print("Single-direction LSTM, decoder_states:", decoder_states)


    elif issubclass(cell_type, keras.layers.GRUCell):
        if bidirectional:
            forward_states = encoder_states[:n_stacks]
            backward_states = encoder_states[n_stacks:]
            b = backward_states
            f = forward_states
            decoder_states = [keras.layers.Concatenate()([f[i], b[i]]) for i in range(0, len(backward_states), 1)]
        else:
            decoder_states = encoder_states

    return decoder_states


In [ ]:
def get_model(
    units: int,
    n_tokens: int,
    n_labels: int,
    n_stacks: int,
    embedding_dim: int,
    cell_type: type[keras.layers.Layer],
    bidirectional: bool = False,
    name: str | None = None,
    max_len_rus: int = None,
    max_len_eng: int = None,
    mask_zero: bool = False,
    attention_dropout: float = 0.0,
    attention: bool = False,

) -> keras.Model:

    '''Creates a model with RNN architecture for sequence multilabel classification.

    Arguments:
        units: dimensionality of RNN cells
        n_tokens: number of tokens in the tokenizer dictionary
        n_labels: number of labels to be predicted
        n_stacks: number of RNN cells in the stack (1 -- no stacking)
        bidirectional: whether or not the model is bidirectional
        name: the model name
        cell_type: type of a cell to use, either keras.layers.LSTMCell or keras.layers.GRUCell

    Returns:
        The model'''

    if attention:
      return_sequences=True
    else:
      # при cells keras сам указывает
      return_sequences=True


    # ----- Encoder -----
    encoder_inputs = keras.layers.Input(shape=(None,), dtype="int32", name="encoder_input")
    x = MaskedEmbedding(input_dim=n_tokens, output_dim=embedding_dim, mask_zero=False, name=("embedding_encoder"))(encoder_inputs)
    x = keras.layers.Dropout(0.2)(x)
    # x = keras.layers.Embedding(input_dim=n_tokens, output_dim=embedding_dim, mask_zero=mask_zero, name=("embedding_encoder"))(encoder_inputs)

    cells = [cell_type(units, dropout=0.2, recurrent_dropout=0.1) for _ in range(n_stacks)] if n_stacks > 1 else cell_type(units)

    if bidirectional:
        rnn_layer = keras.layers.Bidirectional(keras.layers.RNN(cells, return_sequences=return_sequences, return_state=True), name="encoder_rnn")
        encoder_outputs, *encoder_states = rnn_layer(x)
        encoder_states = prepare_encoder_states(encoder_states=encoder_states, cell_type=cell_type, n_stacks=n_stacks, bidirectional=bidirectional)


    else:
        rnn_layer = keras.layers.RNN(cells, return_sequences=return_sequences, return_state=True, name="encoder_rnn")
        encoder_outputs, *encoder_states = rnn_layer(x)
        encoder_states = prepare_encoder_states(encoder_states=encoder_states, cell_type=cell_type, n_stacks=n_stacks, bidirectional=bidirectional)

    # ----- Decoder -----
    decoder_inputs = keras.layers.Input(shape=(None,), name="decoder_input")
    y = MaskedEmbedding(input_dim=n_tokens, output_dim=embedding_dim, mask_zero=False, name=("embedding_decoder"))(decoder_inputs)
    y = keras.layers.Dropout(0.2)(y)
    # y = keras.layers.Embedding(input_dim=n_labels, output_dim=embedding_dim, mask_zero=mask_zero, name=("embedding_decoder"))(decoder_inputs)

    if bidirectional:
      cells = [cell_type(units * 2, dropout=0.2, recurrent_dropout=0.1) for _ in range(n_stacks)] if n_stacks > 1 else cell_type(units * 2)

      decoder_rnn = keras.layers.RNN(cells, return_sequences=return_sequences, return_state=True,  name="decoder_rnn")
    else:
      decoder_rnn = keras.layers.RNN(cells, return_sequences=return_sequences, return_state=True, name="decoder_rnn")
    decoder_outputs, *_ = decoder_rnn(y, initial_state=encoder_states)


    if attention:

      attention_layer = keras.layers.AdditiveAttention(
          use_scale=True,
          dropout=attention_dropout,
          name="Attention",
      )

      encoder_mask = keras.layers.Lambda(
          lambda x: tf.cast(tf.not_equal(x, tokenizer.pad_token_id), tf.float32),
          name="Encoder_Mask"
      )(encoder_inputs)


      decoder_mask = keras.layers.Lambda(
          lambda x: tf.cast(tf.not_equal(x, tokenizer.pad_token_id), tf.float32),
          name="Decoder_Mask"
      )(decoder_inputs)
      print(encoder_inputs)
      print(encoder_mask)


      decoder_mask_exp = tf.keras.layers.Lambda(lambda m: tf.expand_dims(m, axis=-1), name="Decoder_Mask_Exp")(decoder_mask)
      encoder_mask_exp = tf.keras.layers.Lambda(lambda m: tf.expand_dims(m, axis=1), name="Encoder_Mask_Exp")(encoder_mask)
      print(decoder_mask_exp)
      print(encoder_mask_exp)


      context_vector = attention_layer(
          [decoder_outputs, encoder_outputs],
          mask=[decoder_mask, encoder_mask_exp]
      )

      decoder_concat = keras.layers.Concatenate(
          axis=-1,
          name="Decoder_Concat")([decoder_outputs, context_vector])

      decoder_concat = keras.layers.Dropout(0.2)(decoder_concat)

      logits = keras.layers.Dense(n_labels, name="Output_Layer")(decoder_concat)
      model = keras.Model(inputs=[encoder_inputs, decoder_inputs], outputs=logits, name="MyModel_With_Attention")
      return model

    else:
      logits = keras.layers.Dense(n_labels, name=get_name("Output", "Layer"))(decoder_outputs) # actibation = NONE
      model = keras.Model(inputs=[encoder_inputs, decoder_inputs], outputs=logits, name="MyModel_Without_Attention")
      return model



In [ ]:
from keras.layers import LSTMCell, GRUCell

units = 128
embedding_dim = 128
n_tokens = len(tokenizer.get_vocab()) # т.к один токенайзер для rus и eng
n_labels = len(tokenizer.get_vocab()) # т.к один токенайзер для rus и eng



models = [
    get_model(units, n_tokens, n_labels, n_stacks=1, embedding_dim=embedding_dim,
              cell_type=keras.layers.GRUCell, bidirectional=True, name="Without_Attention",
              max_len_rus=max_len_inp, max_len_eng=max_len_tar, attention = False),

    get_model(units, n_tokens, n_labels, n_stacks=1, embedding_dim=embedding_dim,
              cell_type=keras.layers.GRUCell, bidirectional=True, name="With_Attention",
              max_len_rus=max_len_inp, max_len_eng=max_len_tar, attention = True),
]


<KerasTensor shape=(None, None), dtype=int32, sparse=False, ragged=False, name=encoder_input>
<KerasTensor shape=(None, None), dtype=float32, sparse=False, ragged=False, name=keras_tensor_83>
<KerasTensor shape=(None, None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor_89>
<KerasTensor shape=(None, 1, None), dtype=float32, sparse=False, ragged=False, name=keras_tensor_92>


Try to add attention to your model (for example [additive attention](https://keras.io/api/layers/attention_layers/additive_attention/)), does it perform better?

# Training


In [ ]:
names = ["Without_Attention", "With_Attention"]
for model, name in zip(models, names):
    model.my_name = name

In [ ]:
def get_callbacks(filepath):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=filepath,
            save_best_only=True,
            monitor='val_loss',
            mode='min',
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            mode='min',
            verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True,
            mode='min',
            verbose=1
        )
    ]


In [ ]:
loss = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=1e-3,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-7
)


models[0].compile(
        optimizer=optimizer,
        loss=loss,
        metrics=['sparse_categorical_accuracy']
    )



In [ ]:
callbacks = get_callbacks(filepath_best_without)

models[0].fit(
            train_dataset_tf,
            epochs=5,
            validation_data=test_dataset_tf,
            callbacks=callbacks,
            steps_per_epoch=steps_per_epoch,
            validation_steps=validation_steps
        )



Epoch 1/5
9780/9780 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - loss: 3.1909 - sparse_categorical_accuracy: 0.0485
Epoch 1: val_loss improved from inf to 1.76027, saving model to /content/drive/MyDrive/Colab Notebooks/NLP/RNN2/models/models_weight/best_model_rnn2_without_att.keras
9780/9780 ━━━━━━━━━━━━━━━━━━━━ 1133s 116ms/step - loss: 3.1908 - sparse_categorical_accuracy: 0.0485 - val_loss: 1.7603 - val_sparse_categorical_accuracy: 0.0711 - learning_rate: 0.0010
Epoch 2/5
9780/9780 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - loss: 1.6571 - sparse_categorical_accuracy: 0.0725
Epoch 2: val_loss improved from 1.76027 to 1.45143, saving model to /content/drive/MyDrive/Colab Notebooks/NLP/RNN2/models/models_weight/best_model_rnn2_without_att.keras
9780/9780 ━━━━━━━━━━━━━━━━━━━━ 1117s 114ms/step - loss: 1.6571 - sparse_categorical_accuracy: 0.0725 - val_loss: 1.4514 - val_sparse_categorical_accuracy: 0.0771 - learning_rate: 0.0010
Epoch 3/5
9780/9780 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - loss: 1.3644 - sp

In [ ]:
loss = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=1e-3,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-7
)



models[1].compile(
        optimizer=optimizer,
        loss=loss,
        metrics=['sparse_categorical_accuracy']
    )



In [ ]:
callbacks = get_callbacks(filepath_best_with)

models[1].fit(
            train_dataset_tf,
            epochs=5,
            validation_data=test_dataset_tf,
            callbacks=callbacks,
            steps_per_epoch=steps_per_epoch,
            validation_steps=validation_steps
        )



Epoch 1/5
9780/9780 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - loss: 2.9407 - sparse_categorical_accuracy: 0.0529
Epoch 1: val_loss improved from inf to 1.32201, saving model to /content/drive/MyDrive/Colab Notebooks/NLP/RNN2/models/models_weight/best_model_rnn2_with_att.keras
9780/9780 ━━━━━━━━━━━━━━━━━━━━ 1577s 160ms/step - loss: 2.9406 - sparse_categorical_accuracy: 0.0529 - val_loss: 1.3220 - val_sparse_categorical_accuracy: 0.0806 - learning_rate: 0.0010
Epoch 2/5
9780/9780 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - loss: 1.3328 - sparse_categorical_accuracy: 0.0795
Epoch 2: val_loss improved from 1.32201 to 1.09339, saving model to /content/drive/MyDrive/Colab Notebooks/NLP/RNN2/models/models_weight/best_model_rnn2_with_att.keras
9780/9780 ━━━━━━━━━━━━━━━━━━━━ 1553s 159ms/step - loss: 1.3328 - sparse_categorical_accuracy: 0.0795 - val_loss: 1.0934 - val_sparse_categorical_accuracy: 0.0851 - learning_rate: 0.0010
Epoch 3/5
9780/9780 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - loss: 1.1094 - sparse_c

# Testing

In [ ]:
def translate(
    text: str,
    tokenizer: Tokenizer,
    model: keras.Model,
    max_len_eng: int = 79,
    max_len_rus: int = 80
) -> str:

    '''Predicts `text` translation using the `model`.'''

    enc_tokens = tokenizer(
        text,
        max_length=max_len_rus,
        truncation=True,
        return_tensors='np',
        add_special_tokens=False
    )['input_ids'][0]


    encoder_input = tf.convert_to_tensor([enc_tokens], dtype=tf.int32)

    # Энкодер
    embedding_layer = model.get_layer("embedding_encoder")
    encoder_embedded = embedding_layer(encoder_input)
    encoder_outputs, *encoder_states = model.get_layer("encoder_rnn")(encoder_embedded)

    encoder_states = prepare_encoder_states(
        encoder_states,
        cell_type=keras.layers.GRUCell,
        n_stacks=1,
        bidirectional=True
    )

    # Декодер
    target_seq = tf.constant([[tokenizer.bos_token_id]], dtype=tf.int32)
    output_tokens = []

    for step in range(max_len_eng):
        decoder_embedded = model.get_layer("embedding_decoder")(target_seq)
        decoder_outputs, *_ = model.get_layer("decoder_rnn")(decoder_embedded, initial_state=encoder_states)

        if "With_Attention" in model.name:

          encoder_mask = tf.cast(encoder_input != tokenizer.pad_token_id, tf.float32)
          decoder_mask = tf.cast(target_seq != tokenizer.pad_token_id, tf.float32)
          encoder_mask_exp = tf.expand_dims(encoder_mask, axis=1)  # [B, 1, T_enc]


          context_vector = model.get_layer("Attention")(
                [decoder_outputs, encoder_outputs],
                mask=[decoder_mask, encoder_mask_exp]
            )

          decoder_outputs = model.get_layer("Decoder_Concat")([decoder_outputs, context_vector])

        logits = model.get_layer("Output_Layer")(decoder_outputs)
        predicted_id = tf.argmax(logits[:, -1, :], axis=-1).numpy()[0]
        output_tokens.append(predicted_id)

        if predicted_id == tokenizer.eos_token_id:
            break

        target_seq = tf.concat([target_seq, [[predicted_id]]], axis=-1)


    translated_text = tokenizer.decode(output_tokens)
    return translated_text


In [ ]:
example_texts = [
    "Я иду домой.",
    "Как прекрасен этот мир, посмотри!",
    "Книга на столе интересная.",
    "Сегодня хорошая погода.",
    "Я люблю читать книги вечером.",
    "Он играет на гитаре каждую неделю.",
    "Машина стоит у дома.",
    "Птицы поют на деревьях.",
    "Мы поехали в отпуск летом.",
    "Она купила новую сумку.",
    "Кошка спит на диване.",
    "Дети играют во дворе.",
    "Солнце садится за горизонт.",
    "Он работает в большой компании.",
    "Я приготовил ужин для всей семьи.",
    "Подарок оказался очень ценным.",
    "Мы гуляли по парку вечером.",
    "Я забыл ключи дома.",
    "Они обсуждают планы на выходные.",
    "Музыка звучит в комнате.",
    "Я встретил старого друга на улице.",
]

In [ ]:
print(f"\nTranslations for model: {models[0].my_name}")
for text in example_texts:
        translation = translate(
              text=text,
              tokenizer=tokenizer,
              model=models[0],
              max_len_eng=max_len_tar,
              max_len_rus=max_len_inp
          )

        translation = translation.replace("<|endoftext|>", "").strip()
        print(f"Input (RU): {text}")
        print(f"Output (EN): {translation}")


Translations for model: Without_Attention
Input (RU): Я иду домой.
Output (EN): I'm going home.
Input (RU): Как прекрасен этот мир, посмотри!
Output (EN): How fasten these are you crossing!
Input (RU): Книга на столе интересная.
Output (EN): The book is in the center of the book.
Input (RU): Сегодня хорошая погода.
Output (EN): Today is a good weather.
Input (RU): Я люблю читать книги вечером.
Output (EN): I like reading books.
Input (RU): Он играет на гитаре каждую неделю.
Output (EN): He plays the guitar every week.
Input (RU): Машина стоит у дома.
Output (EN): The car is at home.
Input (RU): Птицы поют на деревьях.
Output (EN): The birds are singing in the river.
Input (RU): Мы поехали в отпуск летом.
Output (EN): We went to the summer vacation.
Input (RU): Она купила новую сумку.
Output (EN): She bought a new bag.
Input (RU): Кошка спит на диване.
Output (EN): The cat sleeps on the couch.
Input (RU): Дети играют во дворе.
Output (EN): The children are playing in the yard.
Input (R

In [ ]:
print(f"\nTranslations for model: {models[1].my_name}")
for text in example_texts:
        translation = translate(
              text=text,
              tokenizer=tokenizer,
              model=models[1],
              max_len_eng=max_len_tar,
              max_len_rus=max_len_inp
          )

        translation = translation.replace("<|endoftext|>", "").strip()
        print(f"Input (RU): {text}")
        print(f"Output (EN): {translation}")


Translations for model: With_Attention
Input (RU): Я иду домой.
Output (EN): I'm going home.
Input (RU): Как прекрасен этот мир, посмотри!
Output (EN): How beautiful this is the world!
Input (RU): Книга на столе интересная.
Output (EN): The book is interesting.
Input (RU): Сегодня хорошая погода.
Output (EN): Today is a good weather.
Input (RU): Я люблю читать книги вечером.
Output (EN): I like reading books tonight.
Input (RU): Он играет на гитаре каждую неделю.
Output (EN): He plays the guitar every week.
Input (RU): Машина стоит у дома.
Output (EN): The car cost a house.
Input (RU): Птицы поют на деревьях.
Output (EN): Birds sing on trees.
Input (RU): Мы поехали в отпуск летом.
Output (EN): We went to the vacation.
Input (RU): Она купила новую сумку.
Output (EN): She bought a new bag.
Input (RU): Кошка спит на диване.
Output (EN): The cat sleeps on the couch.
Input (RU): Дети играют во дворе.
Output (EN): Children play the yard.
Input (RU): Солнце садится за горизонт.
Output (EN): 